In [ ]:
# install transformer
!pip install -q \
  transformers==4.51.3 \
  accelerate \
  lm-format-enforcer

In [ ]:
# check GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
from lmformatenforcer import RegexParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

True
Tesla T4


In [ ]:
# load WIKISQL train data
import json
import pprint

file_path = "/content/wikisql_train_preprocessed.json"
with open(file_path, "r") as f:
    data_wikisql = json.load(f)
print("Data loaded successfully!")

# check data structure
print("WIKISQL data num: ", len(data_wikisql))

# first data
# print("\nfirst data:")
# pprint.pprint(data[0])

Data loaded successfully!
WIKISQL data num:  56355


In [ ]:
# Random select data from WIKISQL
import random
def sample_data(data, sample_size):
    random.seed(42)
    sampled_data = random.sample(data, sample_size)
    print("Sampled:", len(sampled_data))
    return sampled_data

In [ ]:
# format schema from WIKISQL
def format_schema_from_wikisql(schema):
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]
    column_types = schema["column_types"]

    table_to_columns = {i: [] for i in range(len(table_names))}

    for idx, (table_id, col_name) in enumerate(column_names):
        col_type = column_types[idx]
        table_to_columns[table_id].append((col_name, col_type))

    lines = []
    for table_id, table_name in enumerate(table_names):
        lines.append(f"Table {table_name}:")
        for col_name, col_type in table_to_columns[table_id]:
            lines.append(f'''- "{col_name}" ({col_type})''')

    return "\n".join(lines)

# example = data[0]
# example_schema_text = format_schema_from_wikisql(example["schema"])
# print(example_schema_text)

# build prompt
def build_prompt(example):
    schema_text = format_schema_from_wikisql(example["schema"])
    question = example["question"]

    prompt = f"""You are a text-to-SQL model.
Convert the natural language question into a SQL query based on the given schema.

Rules:
- Output only SQL.
- Do not explain.
- Do not include markdown code fences.
- Use only the table and column names provided in the schema.

Schema:
{schema_text}

Question:
{question}

SQL:
"""
    return prompt

# example_prompt = build_prompt(example)
# print(example_prompt)

In [ ]:
# load model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer_Qwen = AutoTokenizer.from_pretrained(model_name)

model_Qwen = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Qwen Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen Model loaded successfully.


In [18]:
# LM Format Enforcer
def prepare_lmfe_for_wikisql(tokenizer):
    sql_regex = r"SELECT ([A-Za-z0-9_/ ]+|(?:MAX|MIN|COUNT|SUM|AVG)\([A-Za-z0-9_*/ ]+\)) FROM [A-Za-z0-9_]+( WHERE [A-Za-z0-9_/ ]+ (=|>|<) .+)?"

    parser = RegexParser(sql_regex)
    prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)

    return prefix_fn

import re
def escape_regex_text(s):
    return re.escape(s)
# add Schema Constraints
def prepare_lmfe_for_wikisql2(data, tokenizer):
    table_names = data["schema"]["table_names_original"]
    column_names = [col_name for _, col_name in data["schema"]["column_names_original"]]

    table_pattern = "(" + "|".join(escape_regex_text(t) for t in table_names) + ")"
    column_pattern = "(" + "|".join(escape_regex_text(c) for c in column_names) + ")"
    select_expr_pattern = (rf"({column_pattern}|(?:MAX|MIN|COUNT|SUM|AVG)\((?:{column_pattern}|\*)\))")

    sql_regex = (
        rf"SELECT {select_expr_pattern} "
        rf"FROM {table_pattern}"
        rf"( WHERE {column_pattern} (=|>|<) [^\n]+)?"
    )
    print(sql_regex)

    parser = RegexParser(sql_regex)
    prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)

    return prefix_fn

def prepare_lmfe_for_spider(tokenizer):
    sql_regex = (
        r"(SELECT|WITH) [A-Za-z0-9_.*,()/ \n-]+ "
        r"FROM [A-Za-z0-9_.,() \n-]+"
        r"( WHERE [A-Za-z0-9_.*=<>!()'\"% \n-]+)?"
        r"( GROUP BY [A-Za-z0-9_.,() \n-]+)?"
        r"( HAVING [A-Za-z0-9_.*=<>!()'\"% \n-]+)?"
        r"( ORDER BY [A-Za-z0-9_.,() \n-]+( ASC| DESC)?)?"
        r"( LIMIT [0-9]+)?"
    )
    parser = RegexParser(sql_regex)
    prefix_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)

    return prefix_fn


In [ ]:
# Clean SQL
def clean_sql(text):
    text = text.strip()
    text = text.replace("```sql", "").replace("```", "").strip()
    prefixes = [
        "SQL:",
        "The SQL query is:",
        "Here is the SQL query:",
        "Answer:",
    ]
    for p in prefixes:
        if text.startswith(p):
            text = text[len(p):].strip()

    stop_words = ["Explanation", "Note:", "\n\n"]
    for w in stop_words:
        if w in text:
            text = text.split(w)[0].strip()

    if ";" in text:
        text = text.split(";")[0] + ";"

    # Remove line breaks and extra spaces
    text = text.replace("\n", " ")
    text = " ".join(text.split())

    return text

In [16]:
# Generate SQL
def generate_sql(data, model, tokenizer, prefix_fn, max_new_tokens=100):
    prompt = build_prompt(data)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            prefix_allowed_tokens_fn=prefix_fn
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred_sql = clean_sql(pred_sql)
    return pred_sql

# add Schema Constraints
# generate prefix_fn for every data
def generate_sql2(data, model, tokenizer, max_new_tokens=100):
    prompt = build_prompt(data)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    prefix_fn = prepare_lmfe_for_wikisql2(data, tokenizer)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            prefix_allowed_tokens_fn=prefix_fn
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred_sql = clean_sql(pred_sql)
    return pred_sql

In [ ]:
def instance_id_key(item):
    return int(item["instance_id"].split("_")[-1])
def sorted_result(items):
    results = sorted(items, key=instance_id_key)
    return results
# Save Results
def save_file(output_path, results):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print("Saved to:", output_path)

In [ ]:
# Qwen_Instruct Model
# wikisql Data
# main
# Run items and add the prediction results back to the original structure
from copy import deepcopy
# sample data
sample_size = 5
sampled_data_wikisql = sample_data(data_wikisql, sample_size)

# LMFN constraint
prefix_fn = prepare_lmfe_for_wikisql(tokenizer_Qwen)

# generate sql
results = []
for i, ex in enumerate(sampled_data_wikisql, 1):
    ex_copy = deepcopy(ex)
    ex_copy["pred_sql"] = generate_sql(data = ex_copy, model = model_Qwen, tokenizer = tokenizer_Qwen, prefix_fn = prefix_fn)
    results.append(ex_copy)

    if i % 50 == 0:
        print(f"Done {i}/{sample_size}")
# sort result
sorted_results = sorted_result(results)
pprint.pprint(sorted_results)
# save to file
output_path = f"/content/wikisql_train_sample{sample_size}_constrained_prediction.json"
save_file(output_path, sorted_results)

Sampled: 5
[{'db': '1-12159115-2',
  'gold_sql_query': 'SELECT COUNT(Title) FROM table WHERE Production code = '
                    "'2T6705'",
  'instance_id': 'wikisql_train_1639',
  'pred_sql': "SELECT COUNT(*) FROM table WHERE Production_code = '2T6705' "
              'GROUP BY Production_code;',
  'question': "What's the total number of episodes with production code "
              '2T6705?',
  'schema': {'column_names_original': [[0, 'Series #'],
                                       [0, 'Title'],
                                       [0, 'Directed by'],
                                       [0, 'Written by'],
                                       [0, 'Original air date'],
                                       [0, 'Production code'],
                                       [0, 'U.S. viewers (millions)']],
             'column_types': ['real',
                              'text',
                              'text',
                              'text',
                   

In [19]:
# generate sql
results2 = []
for i, ex in enumerate([sampled_data_wikisql[0]], 1):
    ex_copy = deepcopy(ex)
    ex_copy["pred_sql"] = generate_sql2(data = ex_copy, model = model_Qwen, tokenizer = tokenizer_Qwen)
    results2.append(ex_copy)

    if i % 50 == 0:
        print(f"Done {i}/{sample_size}")
# sort result
sorted_results2 = sorted_result(results2)
pprint.pprint(sorted_results2)
# save to file
output_path = f"/content/wikisql_train2_constrained_prediction.json"
save_file(output_path, sorted_results)

SELECT ((CERCLIS\ ID|Name|County|Proposed|Listed)|(?:MAX|MIN|COUNT|SUM|AVG)\((?:(CERCLIS\ ID|Name|County|Proposed|Listed)|\*)\)) FROM (table)( WHERE (CERCLIS\ ID|Name|County|Proposed|Listed) (=|>|<) [^\n]+)?
[{'db': '2-11960788-1',
  'gold_sql_query': 'SELECT County FROM table WHERE CERCLIS ID = '
                    "'scd037405362'",
  'instance_id': 'wikisql_train_41905',
  'pred_sql': "SELECT County FROM table WHERE CERCLIS ID = 'scd037405362' ;",
  'question': 'What county has a CERCLIS ID of scd037405362?',
  'schema': {'column_names_original': [[0, 'CERCLIS ID'],
                                       [0, 'Name'],
                                       [0, 'County'],
                                       [0, 'Proposed'],
                                       [0, 'Listed']],
             'column_types': ['text', 'text', 'text', 'text', 'text'],
             'table_names_original': ['table']}}]
Saved to: /content/wikisql_train2_constrained_prediction.json
